In [ ]:
using CSV
using DataFrames
using PyCall
using Statistics

repo_root = normpath(joinpath(@__DIR__, ".."))

function add_python_rvo2_path!(repo_root::AbstractString)
    build_dir = joinpath(repo_root, "Python-RVO2", "build")
    if !isdir(build_dir)
        @warn "Python-RVO2 build directory not found" build_dir
        return
    end

    candidate_dirs = sort(filter(path -> isdir(path) && startswith(basename(path), "lib."),
                                 readdir(build_dir, join=true)))
    if isempty(candidate_dirs)
        @warn "No built Python-RVO2 library directory was found" build_dir
        return
    end

    py_path = PyVector(pyimport("sys")."path")
    for candidate in reverse(candidate_dirs)
        if !(candidate in py_path)
            pushfirst!(py_path, candidate)
        end
    end
end

add_python_rvo2_path!(repo_root)
include(joinpath(repo_root, "src", "DistributionallyRobust.jl"))
using .DistributionallyRobust


In [ ]:
include(joinpath(repo_root, "scripts", "default_params", "params_drc_data_trajectron.jl"))

epsilon = 0.15
safety_distance = 10.0
test_scene_id = 0
deterministic = true
num_samples = 1
prediction_steps = 2

output_root = joinpath(@__DIR__, "table1_results")
mkpath(output_root)

@info "Configured Table 1 evaluation" test_data_name epsilon start_time_idx safety_distance deterministic num_samples prediction_steps output_root


In [ ]:
function run_single_experiment(
    run_id::Int,
    ego_pos_init_vec::Vector{Float64},
    ego_pos_goal_vec::Vector{Float64};
    output_root::AbstractString=joinpath(@__DIR__, "table1_results"),
    save_state_histories::Bool=false,
)
    target_speed = 2.0
    sim_horizon = 10.0

    scene_loader, controller, w_init, measurement_schedule, target_trajectory, target_speed =
        controller_setup(
            scene_param,
            predictor_param,
            prediction_device=prediction_device,
            cost_param=cost_param,
            cnt_param=cnt_param,
            dtc=dtc,
            run_id=run_id,
            ego_pos_init_vec=ego_pos_init_vec,
            ego_pos_goal_vec=ego_pos_goal_vec,
            target_speed=target_speed,
            sim_horizon=sim_horizon,
            verbose=true,
        )

    result, controller, _, comp_time_list = evaluate(
        scene_loader,
        controller,
        w_init,
        ego_pos_goal_vec,
        target_speed,
        measurement_schedule,
        target_trajectory,
        pos_error_replan,
        safety_distance,
        max_MPC_iters,
        run_id,
    )

    collision_count = result.total_col
    positional_cost = 0.0
    min_distance = Inf

    for w in result.w_history
        robot_pos = get_position(w.e_state)
        positional_cost += norm(robot_pos - ego_pos_goal_vec)^2 * controller.sim_param.dtc

        for ap_pos in values(w.ap_dict)
            distance = norm(robot_pos - ap_pos)
            if distance < min_distance
                min_distance = distance
            end
        end
    end

    tol_goal = 0.4
    final_robot_pos = get_position(result.w_history[end].e_state)
    reached_goal = norm(final_robot_pos - ego_pos_goal_vec) <= tol_goal
    is_success = (collision_count == 0.0) && reached_goal

    avg_computation_time_ms = mean(comp_time_list) * 1000
    total_sim_time = dtc * length(result.w_history)

    experiment_dir = joinpath(output_root, "experiment_$(run_id)")
    mkpath(experiment_dir)

    summary_df = DataFrame(
        collision_num=[collision_count],
        success=[is_success],
        minimum_distance=[min_distance],
        positional_cost=[positional_cost],
        avg_computation_time_ms=[avg_computation_time_ms],
        sim_time=[total_sim_time],
    )
    CSV.write(joinpath(experiment_dir, "experiment_summary.csv"), summary_df)

    if save_state_histories
        position_history = DataFrame(
            x=[get_position(w.e_state)[1] for w in result.w_history],
            y=[get_position(w.e_state)[2] for w in result.w_history],
        )
        velocity_history = DataFrame(
            vx=[get_velocity(w.e_state)[1] for w in result.w_history],
            vy=[get_velocity(w.e_state)[2] for w in result.w_history],
        )
        CSV.write(joinpath(experiment_dir, "position_history.csv"), position_history)
        CSV.write(joinpath(experiment_dir, "velocity_history.csv"), velocity_history)
    end

    return (
        collision_count=collision_count,
        is_success=is_success,
        min_distance=min_distance,
        positional_cost=positional_cost,
        avg_computation_time_ms=avg_computation_time_ms,
        total_sim_time=total_sim_time,
    )
end

function run_table1_suite(;
    num_experiments::Int=300,
    experiment_states_path::AbstractString=joinpath(@__DIR__, "experiment_states.csv"),
    output_root::AbstractString=joinpath(@__DIR__, "table1_results"),
    save_state_histories::Bool=false,
)
    data = CSV.read(experiment_states_path, DataFrame)
    if nrow(data) < num_experiments
        error("experiment_states.csv does not contain enough rows for the requested number of experiments")
    end

    all_collision = Float64[]
    all_successes = Int[]
    all_min_distances = Float64[]
    all_positional_costs = Float64[]
    all_computation_times = Float64[]
    all_total_sim_time = Float64[]

    for run_id in 1:num_experiments
        global ego_pos_init_vec = [data.Initial_X[run_id], data.Initial_Y[run_id]]
        global ego_pos_goal_vec = [data.Goal_X[run_id], data.Goal_Y[run_id]]

        @info "Running experiment" run_id ego_pos_init_vec ego_pos_goal_vec

        include(joinpath(repo_root, "scripts", "parameter_setup_drc.jl"))

        metrics = run_single_experiment(
            run_id,
            ego_pos_init_vec,
            ego_pos_goal_vec;
            output_root=output_root,
            save_state_histories=save_state_histories,
        )

        push!(all_collision, metrics.collision_count)
        push!(all_successes, metrics.is_success ? 1 : 0)
        push!(all_min_distances, metrics.min_distance)
        push!(all_positional_costs, metrics.positional_cost)
        push!(all_computation_times, metrics.avg_computation_time_ms)
        push!(all_total_sim_time, metrics.total_sim_time)
    end

    collision_num = sum(all_collision)
    total_successes = sum(all_successes)
    avg_min_distance = mean(all_min_distances)
    std_min_distance = std(all_min_distances)
    avg_positional_cost = mean(all_positional_costs)
    std_positional_cost = std(all_positional_costs)
    avg_computation_time = mean(all_computation_times)
    std_computation_time = std(all_computation_times)
    avg_total_sim_time = mean(all_total_sim_time)
    std_total_sim_time = std(all_total_sim_time)

    results_summary = DataFrame(
        collision_num=[collision_num],
        success_count=[total_successes],
        sim_time_mean=[avg_total_sim_time],
        sim_time_std=[std_total_sim_time],
        computation_time_ms_mean=[avg_computation_time],
        computation_time_ms_std=[std_computation_time],
        minimum_distance_mean=[avg_min_distance],
        minimum_distance_std=[std_min_distance],
        positional_cost_mean=[avg_positional_cost],
        positional_cost_std=[std_positional_cost],
        sim_time_display=["$(round(avg_total_sim_time, digits=2)) ? $(round(std_total_sim_time, digits=2))"],
        computation_time_display=["$(round(avg_computation_time, digits=2)) ? $(round(std_computation_time, digits=2))"],
        minimum_distance_display=["$(round(avg_min_distance, digits=2)) ? $(round(std_min_distance, digits=2))"],
        positional_cost_display=["$(round(avg_positional_cost, digits=2)) ? $(round(std_positional_cost, digits=2))"],
    )

    test_data_tag = replace(replace(test_data_name, ".pkl" => ""), r"[^A-Za-z0-9._-]" => "_")
    summary_path = joinpath(output_root, "results_summary_rmv_$(epsilon)_$(test_data_tag).csv")
    CSV.write(summary_path, results_summary)

    @info "Saved Table 1 summary" summary_path
    return results_summary
end


In [ ]:
results_summary = run_table1_suite()
results_summary